# UCI Online Retail II 일매출 Chronos 실험

거래수량과 단가로 일별 순상품매출 추정치를 만들고 Chronos의 마지막 28일 예측을 확인합니다. 출처: UCI Online Retail II, DOI 10.24432/C5CG6D, CC BY 4.0.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from chronos_sales.config import load_config
from chronos_sales.online_retail import load_online_retail_revenue

config = load_config(PROJECT_ROOT / 'configs' / 'online-retail-revenue.yaml')
data_dir = PROJECT_ROOT / 'data' / 'online_retail_revenue'
output_dir = PROJECT_ROOT / 'outputs' / 'online_retail_revenue' / 'zero_shot'

## 변환 감사와 일매출

In [ ]:
audit = pd.read_csv(data_dir / 'transaction_audit.csv')
daily = pd.read_parquet(data_dir / 'daily_revenue.parquet')
display(audit)
display(daily.head())
ax = daily.plot(x='date', y=['gross_revenue_gbp', 'net_revenue_gbp'], figsize=(14, 5), alpha=.8)
ax.set(title='Daily product revenue proxy', ylabel='GBP', xlabel='Date')
plt.show()

## 학습·검증·시험 분할

In [ ]:
manifest = pd.read_csv(data_dir / 'online_retail_manifest.csv')
display(manifest)
dataset = load_online_retail_revenue(data_dir / 'daily_revenue.parquet')
print('전체 일수:', len(dataset[0]['target']), '학습 일수:', len(dataset[0]['target']) - 56)

## 제로샷과 7일 계절 나이브 비교

In [ ]:
comparison_path = output_dir / 'baseline_comparison_metrics.csv'
forecast_path = output_dir / 'zero_shot_test_business_forecasts.csv'
if comparison_path.exists():
    display(pd.read_csv(comparison_path))
if forecast_path.exists():
    forecast = pd.read_csv(forecast_path, parse_dates=['date'])
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(forecast['date'], forecast['actual_revenue'], color='black', label='Actual')
    ax.plot(forecast['date'], forecast['pred_q50'], label='Chronos q50')
    ax.fill_between(forecast['date'], forecast['pred_q10'], forecast['pred_q90'], alpha=.25, label='q10-q90')
    ax.set(title='Last 28 days: revenue forecast', ylabel='GBP')
    ax.legend()
    plt.show()

## 해석 제한
이 데이터는 영국 온라인 소매업체 한 곳의 거래입니다. 세금·배송료·결제 수수료가 없으므로 회계상 순매출이 아닙니다. 경진대회에서는 기술 검증용 공개 데이터로 표시하고 실제 도입 시 국내 매장 POS 데이터로 교체합니다.